In [ ]:
import duckdb, pandas as pd
DB_PATH = r"C:\Users\pc\Desktop\ucode\project\storage\duckdb\options_history.duckdb"
con = duckdb.connect(DB_PATH)

In [ ]:
tables = con.execute("SHOW TABLES").fetchdf()
tables

In [ ]:
for tbl in tables["name"]:
    cnt = con.execute(f'SELECT count(*) FROM "{tbl}"').fetchone()[0]
    print(f"\n{'='*60}")
    print(f"{tbl}  ({cnt:,} rows)")
    print(f"{'='*60}")
    
    cols = con.execute(f"DESCRIBE \"{tbl}\"").fetchdf()
    display(cols)
    
    if cnt > 0:
        dt_cols = [c for c in cols["column_name"] if "time" in c.lower() or "date" in c.lower()]
        for c in dt_cols:
            r = con.execute(f'SELECT min({c}), max({c}) FROM "{tbl}"').fetchone()
            print(f"  {c}: {r[0]}  ->  {r[1]}")
        
        for c in cols["column_name"]:
            n = con.execute(f'SELECT count(*) FROM "{tbl}" WHERE {c} IS NULL').fetchone()[0]
            if n > 0:
                print(f"  WARNING: {c}: {n:,} NULLs")

con.close()

In [ ]:
# Quick stats on options_data_clean
con = duckdb.connect(DB_PATH)
print("=== CALL WEEK atm_distance=0 ===")
df = con.execute("""
    SELECT count(*) as rows, min(timestamp) as first, max(timestamp) as last,
           count(DISTINCT strike) as strikes, count(DISTINCT date(timestamp)) as days
    FROM options_data_clean
    WHERE option_type='CALL' AND expiry_code=1 AND expiry_flag='WEEK' AND atm_distance=0
""").fetchdf()
display(df)

print("\n=== CALL WEEK all strikes ===")
df2 = con.execute("""
    SELECT count(*) as rows, min(timestamp), max(timestamp),
           count(DISTINCT strike) as strikes
    FROM options_data_clean
    WHERE option_type='CALL' AND expiry_code=1 AND expiry_flag='WEEK'
""").fetchdf()
display(df2)

con.close()

In [ ]:
# Sample recent data
con = duckdb.connect(DB_PATH)
df = con.execute("""
    SELECT timestamp, close, strike
    FROM options_data_clean
    WHERE option_type='CALL' AND expiry_code=1 AND expiry_flag='WEEK' AND atm_distance=0
    ORDER BY timestamp DESC LIMIT 10
""").fetchdf()
display(df)
con.close()